# VIP Glia Mutation App - Standalone Test Launcher

This notebook launches the copied mutation app from inside `Digifly Public` without relying on the older `Digifly_NEW/VIP_Glia_Sim` folder.

Use it as the public smoke-test launcher for the app import.

What it does:

- resolves the app root from this copied public folder
- resolves Phase 2 from the surrounding `Phase 2` folder, unless `DIGIFLY_PHASE2_ROOT` is set
- resolves SWC data from `DIGIFLY_SWC_DIR`, or `Phase 2/data/export_swc` by default
- checks that the app script and Python runtime are present
- builds the standalone command
- launches the desktop app from a final explicit cell


In [ ]:
from __future__ import annotations

import os
import shlex
import subprocess
import sys
import time
from pathlib import Path


def _find_app_root() -> Path:
    candidates = []
    env_root = os.environ.get('DIGIFLY_VIP_GLIA_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))
    try:
        candidates.append(Path.cwd())
    except Exception:
        pass
    try:
        cwd = Path.cwd()
        for base in [cwd] + list(cwd.parents):
            candidates.append(base / 'Phase 2' / 'apps' / 'VIP_Glia_Sim')
            candidates.append(base / 'apps' / 'VIP_Glia_Sim')
    except Exception:
        pass
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        for root in [candidate] + list(candidate.parents):
            if (root / 'tools' / 'morphology_mutation_app.py').exists():
                return root
    raise RuntimeError('Could not locate VIP_Glia_Sim app root. Set DIGIFLY_VIP_GLIA_ROOT.')


def _resolve_python_bin() -> Path:
    candidates = [
        os.environ.get('VIP_PYTHON_BIN'),
        '/opt/anaconda3/bin/python3.12',
        '/opt/anaconda3/bin/python3',
        '/opt/anaconda3/bin/python',
        sys.executable,
    ]
    for cand in candidates:
        if not cand:
            continue
        p = Path(str(cand)).expanduser()
        if p.exists():
            return p.resolve()
    return Path(sys.executable).resolve()


APP_ROOT = _find_app_root()
PHASE2_ROOT = Path(os.environ.get('DIGIFLY_PHASE2_ROOT', str(APP_ROOT.parents[1]))).expanduser().resolve()
SWC_DIR = Path(os.environ.get('DIGIFLY_SWC_DIR', str(PHASE2_ROOT / 'data' / 'export_swc'))).expanduser().resolve()
APP_PATH = APP_ROOT / 'tools' / 'morphology_mutation_app.py'
PYTHON_BIN = _resolve_python_bin()
OUTPUT_ROOT = APP_ROOT / 'notebooks' / 'debug' / 'outputs'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print('APP_ROOT   =', APP_ROOT)
print('PHASE2_ROOT=', PHASE2_ROOT)
print('SWC_DIR    =', SWC_DIR)
print('APP_PATH   =', APP_PATH)
print('PYTHON_BIN =', PYTHON_BIN)
print('OUTPUT_ROOT=', OUTPUT_ROOT)


In [ ]:
# Dependency and path check. This does not launch the desktop app.
required_paths = {
    'app script': APP_PATH,
    'app root': APP_ROOT,
    'Phase 2 root': PHASE2_ROOT,
}
missing = [name for name, path in required_paths.items() if not Path(path).exists()]
if missing:
    raise FileNotFoundError('Missing required paths: ' + ', '.join(missing))

optional_missing = []
if not SWC_DIR.exists():
    optional_missing.append(f'SWC_DIR does not exist yet: {SWC_DIR}')

for package_name in ['numpy', 'pandas', 'pyvista']:
    probe = subprocess.run(
        [str(PYTHON_BIN), '-c', f'import {package_name}; print({package_name}.__name__)'],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
    )
    if probe.returncode != 0:
        optional_missing.append(f'{package_name} import failed for {PYTHON_BIN}: {probe.stderr.strip()}')

if optional_missing:
    print('Warnings:')
    for msg in optional_missing:
        print(' -', msg)
else:
    print('Dependency/path check passed.')


In [ ]:
# Test inputs. Edit these for the SWCs you want to load.
# If SWC_DIR is absent in this public repo, set DIGIFLY_SWC_DIR before launching Jupyter
# or edit SWC_DIR above to point at a real Phase 1 export_swc folder.
NEURON_IDS = [10000, 10002, 10068, 10110, 11446, 11654]
MUTATION_TAG = 'standalone_test'

# Rendering and app defaults.
RENDER_MODE = 'neuroglancer'       # 'tube', 'skeleton', or 'neuroglancer'
NEUROGLANCER_QUALITY = 'ultra'     # 'auto', 'balanced', 'high', 'ultra'
START_SOLO = True
START_NEURON_ID = int(NEURON_IDS[0]) if NEURON_IDS else None
SKELETON_LINE_WIDTH = 6.0

# Mutation defaults.
THIN_FACTOR = 0.8
THICK_FACTOR = 1.2
GROW_LENGTH_UM = 18.0
GROW_SEGMENTS = 4
GROW_RADIUS_SCALE = 0.85

# Optional flow movie inputs. Leave FLOW_RUN_DIR=None for plain mutation testing.
FLOW_RUN_DIR = None
FLOW_FOCUS_PAIR = None
FLOW_FPS = 20
FLOW_FRAME_STRIDE = 4
FLOW_SPEED_UM_PER_MS = 220.0
FLOW_PULSE_SIGMA_MS = 0.6
FLOW_THRESHOLD_MV = 0.0
FLOW_MAX_MS = 25.0


In [ ]:
cmd = [
    str(PYTHON_BIN),
    str(APP_PATH),
    '--swc-dir', str(SWC_DIR),
    '--phase2-root', str(PHASE2_ROOT),
    '--neuron-ids', ','.join(str(int(x)) for x in NEURON_IDS),
    '--output-root', str(OUTPUT_ROOT),
    '--tag', str(MUTATION_TAG),
    '--thin-factor', str(THIN_FACTOR),
    '--thick-factor', str(THICK_FACTOR),
    '--grow-length-um', str(GROW_LENGTH_UM),
    '--grow-segments', str(GROW_SEGMENTS),
    '--grow-radius-scale', str(GROW_RADIUS_SCALE),
    '--render-mode', str(RENDER_MODE),
    '--skeleton-line-width', str(SKELETON_LINE_WIDTH),
    '--visual-style', 'classic',
    '--neuroglancer-quality', str(NEUROGLANCER_QUALITY),
]
if START_SOLO:
    cmd.append('--start-solo')
if START_NEURON_ID is not None:
    cmd.extend(['--start-neuron-id', str(int(START_NEURON_ID))])
if FLOW_RUN_DIR is not None:
    cmd.extend(['--flow-run-dir', str(Path(FLOW_RUN_DIR).expanduser().resolve())])
if FLOW_FOCUS_PAIR is not None:
    cmd.extend(['--flow-focus-pair', ','.join(str(int(x)) for x in FLOW_FOCUS_PAIR)])
cmd.extend(['--flow-fps', str(FLOW_FPS)])
cmd.extend(['--flow-frame-stride', str(FLOW_FRAME_STRIDE)])
cmd.extend(['--flow-speed-um-per-ms', str(FLOW_SPEED_UM_PER_MS)])
cmd.extend(['--flow-pulse-sigma-ms', str(FLOW_PULSE_SIGMA_MS)])
cmd.extend(['--flow-threshold-mv', str(FLOW_THRESHOLD_MV)])
cmd.extend(['--flow-max-ms', str(FLOW_MAX_MS)])

print(shlex.join(cmd))


In [ ]:
# Safe CLI smoke test: print help without opening the desktop app.
help_probe = subprocess.run(
    [str(PYTHON_BIN), str(APP_PATH), '--help'],
    cwd=str(APP_ROOT),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print('exit =', help_probe.returncode)
print(help_probe.stdout[:4000])
if help_probe.returncode != 0:
    raise RuntimeError('morphology_mutation_app.py --help failed')


In [ ]:
# Standalone app launch. This opens the desktop PyVista window.
# Close the desktop window when done. Logs are written under notebooks/debug/outputs/_launcher_logs.
LOG_DIR = OUTPUT_ROOT / '_launcher_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
LAUNCH_LOG = LOG_DIR / f'morphology_mutation_standalone_test_{int(time.time())}.log'

env = os.environ.copy()
env['DIGIFLY_VIP_GLIA_ROOT'] = str(APP_ROOT)
env['DIGIFLY_PHASE2_ROOT'] = str(PHASE2_ROOT)
env['DIGIFLY_SWC_DIR'] = str(SWC_DIR)

with LAUNCH_LOG.open('w', encoding='utf-8') as log_file:
    mutation_proc = subprocess.Popen(
        cmd,
        cwd=str(APP_ROOT),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
    )

time.sleep(1.0)
rc = mutation_proc.poll()
if rc is None:
    print(f'Launched mutation app for standalone testing. PID={mutation_proc.pid}')
    print('Launcher log =', LAUNCH_LOG)
else:
    print(f'Launch failed early. exit={rc}')
    print('Launcher log =', LAUNCH_LOG)
    print(LAUNCH_LOG.read_text(encoding='utf-8')[-4000:])
